In [9]:
import pandas as pd

df=pd.read_csv("results/bilan.csv").head(15)

df.head(3)

,question_nb,question,expected_answer,no_rag_code_answer,code_answer,n8n_answer
0,1,What is Docker?,"Docker is an open platform for developing, shi...",Docker is a containerization platform that all...,"Docker is an open platform for developing, shi...","Docker is an open platform for developing, shi..."
1,2,What is a container image?,A container image is a standardized package th...,"A container image is a lightweight, packaged r...",A container image is a standardized package th...,A container image is a standardized package th...
2,3,What is the difference between a container and...,A container is a runnable instance of an image.,A Docker image is a template that defines the ...,"A container is a running instance of an image,...",A container is an isolated process that runs f...


In [10]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder=SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1587.31it/s]


# Cosine Similarity

In [11]:
code_count=0
n8n_count=0
eq=0
sum_code=0
sum_n8n=0

for i, row in df.iterrows():
    code_answer=row["code_answer"]
    embedded_code_answer=embedder.encode(code_answer).reshape(1,-1)

    n8n_answer=row["n8n_answer"]
    embedded_n8n_answer=embedder.encode(n8n_answer).reshape(1,-1)

    expected_answer=row["expected_answer"]
    embedded_expected_answer=embedder.encode(expected_answer).reshape(1,-1)

    code_similarity=cosine_similarity(embedded_expected_answer,embedded_code_answer)[0][0]
    n8n_similarity=cosine_similarity(embedded_expected_answer,embedded_n8n_answer)[0][0]

    sum_code+=code_similarity
    sum_n8n+=n8n_similarity

    if code_similarity> n8n_similarity : code_count+=1
    if n8n_similarity>code_similarity : n8n_count+=1
    if n8n_similarity==code_similarity : eq+=1

    # print("code_similarity",code_similarity)
    # print("n8n_similarity",n8n_similarity)

code_similarity_avg=sum_code/len(df)
n8n_similarity_avg=sum_n8n/len(df)

# print(f"code count : {code_count}/{len(df)}")
# print(f"n8n count : {n8n_count}/{len(df)}")
# print(f" n8n and code eq nb : {eq}/{len(df)}")

print("average code similarity : ",code_similarity_avg)
print("average n8n similarity : ",n8n_similarity_avg)

average code similarity :  0.896831
average n8n similarity :  0.86350936


# BERTScore

In [12]:
from bert_score import score

code_p,code_r,code_f1=score(
    list(df["code_answer"]),
    list(df["expected_answer"]),
    lang="en"
)

n8n_p,n8n_r,n8n_f1=score(
    list(df["n8n_answer"]),
    list(df["expected_answer"]),
    lang="en"
)

print("average code precision score : ",code_p.mean().item())
print("average n8n precision score : ",n8n_p.mean().item())

print("average code recall score : ",code_r.mean().item())
print("average n8n recall score : ",n8n_r.mean().item())

print("average code f1 score : ",code_f1.mean().item())
print("average n8n f1 score : ",n8n_f1.mean().item())

Loading weights: 100%|██████████| 389/389 [00:02<00:00, 192.74it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 243.37it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

average code precision score :  0.9223721623420715
average n8n precision score :  0.9048814177513123
average code recall score :  0.952601969242096
average n8n recall score :  0.9496177434921265
average code f1 score :  0.9370006322860718
average n8n f1 score :  0.9264745116233826
